In [1]:
# Import thu vien
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# Doc du lieu va chon device
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(processed_dir / "train.csv")
valid_df = pd.read_csv(processed_dir / "valid.csv")
test_df = pd.read_csv(processed_dir / "test.csv")
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
train_df.head()

device: cuda


,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,hour,month,...,cloud_cover_lag_2h,cloud_cover_lag_3h,temperature_2m_lag_1h,temperature_2m_lag_2h,temperature_2m_lag_3h,precipitation_rolling_3h,precipitation_rolling_6h,rain_next_1h,rain_next_2h,rain_next_3h
0,2019-01-01 05:00:00,0.0,64,1026.0,8.8,15.0,17,100,5,1,...,34.0,39.0,9.0,9.1,9.7,0.0,0.0,0,0,0
1,2019-01-01 06:00:00,0.0,64,1026.4,8.9,14.6,16,100,6,1,...,59.0,34.0,8.8,9.0,9.1,0.0,0.0,0,0,0
2,2019-01-01 07:00:00,0.0,71,1026.5,9.2,15.8,24,100,7,1,...,100.0,59.0,8.9,8.8,9.0,0.0,0.0,0,0,0
3,2019-01-01 08:00:00,0.0,67,1027.4,9.8,15.6,23,100,8,1,...,100.0,100.0,9.2,8.9,8.8,0.0,0.0,0,0,0
4,2019-01-01 09:00:00,0.0,65,1028.3,10.3,14.8,18,100,9,1,...,100.0,100.0,9.8,9.2,8.9,0.0,0.0,0,0,0


In [3]:
# Tao feature sin cos cho bien chu ky
target_cols = ["rain_next_1h", "rain_next_2h", "rain_next_3h"]

def add_cyclic_features(df):
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)
    df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["wind_dir_sin"] = np.sin(2 * np.pi * df["wind_direction_10m"] / 360)
    df["wind_dir_cos"] = np.cos(2 * np.pi * df["wind_direction_10m"] / 360)
    return df

train_df = add_cyclic_features(train_df)
valid_df = add_cyclic_features(valid_df)
test_df = add_cyclic_features(test_df)
train_valid_df = add_cyclic_features(train_valid_df)

drop_cols = ["datetime", "hour", "month", "dayofweek", "wind_direction_10m"] + target_cols
feature_cols = [col for col in train_df.columns if col not in drop_cols]

print("num_features:", len(feature_cols))
feature_cols

num_features: 28


['precipitation',
 'relative_humidity_2m',
 'surface_pressure',
 'temperature_2m',
 'wind_speed_10m',
 'cloud_cover',
 'precipitation_lag_1h',
 'precipitation_lag_2h',
 'precipitation_lag_3h',
 'relative_humidity_2m_lag_1h',
 'relative_humidity_2m_lag_2h',
 'relative_humidity_2m_lag_3h',
 'cloud_cover_lag_1h',
 'cloud_cover_lag_2h',
 'cloud_cover_lag_3h',
 'temperature_2m_lag_1h',
 'temperature_2m_lag_2h',
 'temperature_2m_lag_3h',
 'precipitation_rolling_3h',
 'precipitation_rolling_6h',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'dayofweek_sin',
 'dayofweek_cos',
 'wind_dir_sin',
 'wind_dir_cos']

In [4]:
# Chuan hoa feature va tao DataLoader
def make_tensors(fit_df, data_df, scaler=None):
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(fit_df[feature_cols])

    x = scaler.transform(data_df[feature_cols]).astype(np.float32)
    y = data_df[target_cols].values.astype(np.float32)
    return x, y, scaler

def make_loader(x, y, shuffle=False, batch_size=512):
    dataset = TensorDataset(torch.tensor(x), torch.tensor(y))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

x_train, y_train, scaler = make_tensors(train_df, train_df)
x_valid, y_valid, _ = make_tensors(train_df, valid_df, scaler)

train_loader = make_loader(x_train, y_train, shuffle=True)
valid_loader = make_loader(x_valid, y_valid)
input_dim = len(feature_cols)

x_train.shape, y_train.shape

((43819, 28), (43819, 3))

In [5]:
# Dinh nghia model va ham train
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim, output_dim=3):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

def loss_on_loader(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    total_rows = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            loss = loss_fn(model(xb), yb)
            total_loss += loss.item() * len(xb)
            total_rows += len(xb)

    return total_loss / total_rows

def train_model(train_loader, input_dim, epochs=80, lr=0.001, valid_loader=None):
    model = LogisticRegressionModel(input_dim).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_loss = float("inf")
    best_state = None
    best_epoch = epochs

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        total_rows = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)
            total_rows += len(xb)

        train_loss = total_loss / total_rows
        valid_loss = np.nan

        if valid_loader is not None:
            valid_loss = loss_on_loader(model, valid_loader, loss_fn)
            if valid_loss < best_loss:
                best_loss = valid_loss
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_epoch

In [6]:
# Train tren train va chon epoch bang valid
torch.manual_seed(42)

epochs = 80
lr = 0.001

valid_model, history_df, best_epoch = train_model(train_loader, input_dim, epochs=epochs, lr=lr, valid_loader=valid_loader)
history_df.to_csv(results_dir / "logistic_regression_training_history.csv", index=False)

print("best_epoch:", best_epoch)
history_df.tail()

best_epoch: 60


,epoch,train_loss,valid_loss
75,76,0.404141,0.413656
76,77,0.404119,0.414094
77,78,0.404129,0.413865
78,79,0.404103,0.414484
79,80,0.404091,0.414323


In [7]:
# Ham du doan xac suat va tinh metric
def predict_proba(model, x):
    model.eval()
    with torch.no_grad():
        x_tensor = torch.tensor(x, dtype=torch.float32).to(device)
        prob = torch.sigmoid(model(x_tensor)).cpu().numpy()
    return prob

def evaluate_targets(y_true, y_prob, y_pred, split_name):
    rows = []

    for i, target in enumerate(target_cols):
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred[:, i], labels=[0, 1]).ravel()
        rows.append({
            "model": "logistic_regression",
            "split": split_name,
            "target": target,
            "threshold": 0.5,
            "log_loss": log_loss(y_true[:, i], y_prob[:, i], labels=[0, 1]),
            "brier_score": brier_score_loss(y_true[:, i], y_prob[:, i]),
            "roc_auc": roc_auc_score(y_true[:, i], y_prob[:, i]),
            "pr_auc": average_precision_score(y_true[:, i], y_prob[:, i]),
            "accuracy": accuracy_score(y_true[:, i], y_pred[:, i]),
            "precision": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "recall": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "f1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        })

    return pd.DataFrame(rows)

In [8]:
# Train lai bang train valid va test final
x_train_valid, y_train_valid, final_scaler = make_tensors(train_valid_df, train_valid_df)
x_test, y_test, _ = make_tensors(train_valid_df, test_df, final_scaler)

train_valid_loader = make_loader(x_train_valid, y_train_valid, shuffle=True)

torch.manual_seed(42)
final_model, _, _ = train_model(train_valid_loader, input_dim, epochs=best_epoch, lr=lr)

test_prob = predict_proba(final_model, x_test)
test_pred = (test_prob >= 0.5).astype(int)
metrics_df = evaluate_targets(y_test, test_prob, test_pred, "test")

test_predictions = test_df[["datetime"]].copy()
for i, target in enumerate(target_cols):
    test_predictions[f"{target}_true"] = y_test[:, i].astype(int)
    test_predictions[f"{target}_prob"] = test_prob[:, i]
    test_predictions[f"{target}_pred"] = test_pred[:, i]

metrics_df.to_csv(results_dir / "logistic_regression_test_metrics.csv", index=False)
test_predictions.to_csv(results_dir / "logistic_regression_test_predictions.csv", index=False)

metrics_df

,model,split,target,threshold,log_loss,brier_score,roc_auc,pr_auc,accuracy,precision,recall,f1,tn,fp,fn,tp
0,logistic_regression,test,rain_next_1h,0.5,0.370876,0.112277,0.889852,0.773071,0.844011,0.787285,0.567938,0.659861,6066,358,1008,1325
1,logistic_regression,test,rain_next_2h,0.5,0.416988,0.132424,0.853114,0.675194,0.809410,0.686937,0.522932,0.593818,5868,556,1113,1220
2,logistic_regression,test,rain_next_3h,0.5,0.421358,0.136014,0.845230,0.655833,0.801188,0.663536,0.514788,0.579773,5815,609,1132,1201
